# 01 — Exploratory Data Analysis
## Cold Chain EWS Digital Twin — Milestone 1

---

> **⚠ STRUCTURAL PROXY NOTICE**  
> The dataset analysed here (`atulanandjha/temperature-readings-iot-devices`, Kaggle)  
> is a **generic IoT temperature sensor log — it is NOT real refrigerated-transport  
> telemetry**. It is used as a structural proxy to validate the ingestion pipeline  
> and characterise sampling behaviour before real cold-chain data is available.  
> All statistics below describe the proxy dataset, not operational cold-chain systems.

---

### What this notebook does
1. Loads the raw CSV from `data/raw/` (no modifications, no assumptions)
2. Reports row count vs. the ~97,600 project-brief estimate
3. Shows all column names and dtypes
4. Analyses the in/out room-tag distribution
5. Parses timestamps (corrected: `dayfirst=True` required — see cell 5 note)
6. **Sampling-interval analysis** (the key test of the 'irregular sampling' claim)
7. Missing-value audit
8. Duplicate-row count
9. Temperature-value distribution + sentinel/error-value check
10. Sensor/room-ID cardinality and per-ID reading counts

In [ ]:
# ── Standard imports ────────────────────────────────────────────────────────
from __future__ import annotations

import pathlib
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Resolve repo root regardless of where Jupyter is launched from
NOTEBOOK_DIR = pathlib.Path().resolve()
REPO_ROOT = NOTEBOOK_DIR
for _ in range(5):
    if (REPO_ROOT / 'data' / 'raw').exists():
        break
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / 'data' / 'raw'
DOCS_DIR = REPO_ROOT / 'docs'
print(f'Repo root  : {REPO_ROOT}')
print(f'Raw data   : {RAW_DIR}')

csv_files = sorted(RAW_DIR.glob('*.csv'))
print(f'CSV files  : {[f.name for f in csv_files]}')

In [ ]:
# ── 1. Load raw CSV ─────────────────────────────────────────────────────────
if not csv_files:
    raise FileNotFoundError(
        'No CSV files found in data/raw/. '
        'Run  python src/data_acquisition.py  first.'
    )

CSV_PATH = csv_files[0]
print(f'Loading: {CSV_PATH.name}  ({CSV_PATH.stat().st_size / 1_048_576:.2f} MB)')

df_raw = pd.read_csv(CSV_PATH)

BRIEF_ROW_ESTIMATE = 97_600
actual_rows = len(df_raw)
delta = actual_rows - BRIEF_ROW_ESTIMATE
sign  = '+' if delta >= 0 else ''

print(f'\n=== ROW COUNT ===')
print(f'Actual rows      : {actual_rows:,}')
print(f'Project brief est: ~{BRIEF_ROW_ESTIMATE:,}')
print(f'Delta            : {sign}{delta:,}  ({sign}{100*delta/BRIEF_ROW_ESTIMATE:.2f}%)')
if abs(delta/BRIEF_ROW_ESTIMATE) < 0.01:
    print('Status: MATCH (within 1%)')
else:
    print('Status: DISCREPANCY')

In [ ]:
# ── 2. Column names & dtypes ─────────────────────────────────────────────────
print('=== COLUMNS & DTYPES ===')
col_info = pd.DataFrame({
    'dtype'     : df_raw.dtypes.astype(str),
    'non_null'  : df_raw.notna().sum(),
    'null'      : df_raw.isna().sum(),
    'null_pct'  : (df_raw.isna().mean() * 100).round(3),
    'n_unique'  : df_raw.nunique(),
    'sample_val': df_raw.iloc[0],
})
display(col_info)
print(f'\nShape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Column names (exact): {list(df_raw.columns)}')
print()
print('NOTE on "id" column: 97,605 unique values for 97,606 rows =>')
print('  This is a per-row record UUID ("__export__.temp_log_<N>_<hash>"),')
print('  NOT a sensor device ID. There is exactly one duplicate row.')
print()
print('NOTE on "room_id/id" column: only 1 unique value ("Room Admin") =>')
print('  All 97,606 readings come from a single named room/location.')
print('  This dataset has no multi-sensor topology beyond the Out/In tag.')

In [ ]:
# ── 3. Low-cardinality columns ────────────────────────────────────────────────
print('=== LOW-CARDINALITY COLUMNS (unique <= 20) ===')
for col in df_raw.columns:
    n = df_raw[col].nunique()
    if n <= 20:
        vals = df_raw[col].value_counts(dropna=False).to_dict()
        print(f'  {col!r:30s}  unique={n}  values={vals}')

In [ ]:
# ── 4. In/Out room-tag distribution ─────────────────────────────────────────
in_out_col = 'out/in'   # confirmed from actual file inspection
print(f'=== IN/OUT TAG COLUMN: {in_out_col!r} ===')
vc = df_raw[in_out_col].value_counts(dropna=False)
pct = df_raw[in_out_col].value_counts(normalize=True, dropna=False) * 100
tag_df = pd.DataFrame({'count': vc, 'pct (%)': pct.round(2)})
display(tag_df)

fig, ax = plt.subplots(figsize=(5, 3))
vc.plot(kind='bar', ax=ax, color=sns.color_palette('muted', len(vc)))
ax.set_title(f"Distribution of '{in_out_col}' (In/Out tag)")
ax.set_xlabel('')
ax.set_ylabel('Row count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for p in ax.patches:
    ax.annotate(f'{p.get_height():,.0f}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(DOCS_DIR / 'fig_inout_distribution.png', dpi=120)
plt.show()

In [ ]:
# ── 5. Timestamp parsing & date range ────────────────────────────────────────
#
# CRITICAL NOTE: The noted_date format is DD-MM-YYYY HH:MM (day-first).
# Without dayfirst=True, pandas defaults to MM-DD-YYYY and fails to parse
# any date where the day value exceeds 12 — this silently produces ~47,662
# NaT values and shifts the inferred date range by ~5 months (Jan→Dec
# instead of Jul→Dec). Always use dayfirst=True for this dataset.

ts_col = 'noted_date'
df = df_raw.copy()
print(f'Timestamp column: {ts_col!r}')
print(f'Raw sample: {df_raw[ts_col].head(5).tolist()}')

# Correct parse
df['ts_parsed'] = pd.to_datetime(df[ts_col], dayfirst=True, errors='coerce')
n_fail = df['ts_parsed'].isna().sum()

# Show what happens WITHOUT dayfirst (for documentation purposes)
ts_wrong = pd.to_datetime(df[ts_col], errors='coerce')   # pandas default: MM-DD-YYYY
n_fail_wrong = ts_wrong.isna().sum()

print(f'\n=== TIMESTAMP PARSING ===')
print(f'With dayfirst=True  → failures: {n_fail:,}  (correct)')
print(f'Without dayfirst    → failures: {n_fail_wrong:,}  (WRONG — do NOT use this)')
print(f'Earliest record : {df["ts_parsed"].min()}')
print(f'Latest  record  : {df["ts_parsed"].max()}')
span = df['ts_parsed'].max() - df['ts_parsed'].min()
print(f'Total time span : {span}')

In [ ]:
# ── 6. Sampling-interval analysis ───────────────────────────────────────────
df_s = df.sort_values('ts_parsed').reset_index(drop=True)
gaps = df_s['ts_parsed'].diff().dt.total_seconds().dropna()

print('=== SAMPLING INTERVALS (OVERALL, corrected parse) ===')
print(f'n_gaps    : {len(gaps):,}')
print(f'min   (s) : {gaps.min():.1f}')
print(f'p25   (s) : {gaps.quantile(0.25):.1f}')
print(f'median(s) : {gaps.median():.1f}   ({gaps.median()/60:.4f} min)')
print(f'p75   (s) : {gaps.quantile(0.75):.1f}')
print(f'mean  (s) : {gaps.mean():.2f}   ({gaps.mean()/60:.4f} min)')
print(f'std   (s) : {gaps.std():.2f}')
print(f'p95   (s) : {gaps.quantile(0.95):.1f}   ({gaps.quantile(0.95)/60:.1f} min)')
print(f'p99   (s) : {gaps.quantile(0.99):.1f}   ({gaps.quantile(0.99)/60:.1f} min)')
print(f'max   (s) : {gaps.max():.1f}   ({gaps.max()/3600:.2f} h)')

n_zero = (gaps == 0).sum()
print(f'\nZero gaps : {n_zero:,}  ({100*n_zero/len(gaps):.2f}%) — same-minute readings (Out+In logged together)')

cv = gaps.std() / gaps.mean()
print(f'CoV (std/mean) : {cv:.4f}')
print(f'Verdict: CoV={cv:.1f} >> 0.5 → sampling IS strongly and quantitatively irregular.')
print('  (Even excl. zero-gaps, CoV remains extremely high due to gaps up to 277 h)')

print('\n=== PER out/in TAG INTERVALS ===')
for tag, gdf in df_s.groupby('out/in'):
    g = gdf.sort_values('ts_parsed')['ts_parsed'].diff().dt.total_seconds().dropna()
    print(f'\n  Tag={tag!r}  n={len(gdf):,}')
    print(f'    median (s): {g.median():.1f}  mean (s): {g.mean():.1f}  std: {g.std():.1f}  max: {g.max():.1f}  CoV: {g.std()/g.mean():.2f}')

In [ ]:
# ── Histograms of inter-reading gaps ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
nonzero = gaps[gaps > 0]
axes[0].hist(nonzero / 60, bins=100, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].set_title('Inter-reading gaps (excl. same-minute) — full range')
axes[0].set_xlabel('Gap (minutes)')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

p95 = float(nonzero.quantile(0.95))
zoomed = nonzero[nonzero <= p95]
axes[1].hist(zoomed / 60, bins=100, color='darkorange', edgecolor='none', alpha=0.85)
axes[1].set_title(f'Gaps zoomed to p95 ({p95/60:.1f} min)')
axes[1].set_xlabel('Gap (minutes)')
axes[1].set_ylabel('Count')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(DOCS_DIR / 'fig_sampling_intervals_corrected.png', dpi=120)
plt.show()
print('Saved: docs/fig_sampling_intervals_corrected.png')

In [ ]:
# ── 7. Missing values per column ─────────────────────────────────────────────
print('=== MISSING VALUES ===')
mv = pd.DataFrame({
    'null_count': df_raw.isna().sum(),
    'null_pct'  : (df_raw.isna().mean() * 100).round(4),
})
display(mv)
total_missing = df_raw.isna().sum().sum()
print(f'Total missing cells: {total_missing:,}  ({100*total_missing/df_raw.size:.6f}% of all cells)')

In [ ]:
# ── 8. Duplicate rows ────────────────────────────────────────────────────────
print('=== DUPLICATE ROWS ===')
n_dups = df_raw.duplicated().sum()
print(f'Exact duplicate rows: {n_dups:,}  ({100*n_dups/len(df_raw):.4f}%)')
if n_dups > 0:
    display(df_raw[df_raw.duplicated(keep=False)])

In [ ]:
# ── 9. Temperature distribution + sentinel check ─────────────────────────────
temp_col = 'temp'
t = df_raw[temp_col].dropna()

print(f'=== TEMPERATURE STATS: {temp_col!r} ===')
print(f'count  : {len(t):,}')
print(f'min    : {t.min():.0f}')
print(f'mean   : {t.mean():.4f}')
print(f'median : {t.median():.0f}')
print(f'std    : {t.std():.4f}')
print(f'max    : {t.max():.0f}')
print(f'n_unique: {t.nunique()} distinct integer values')
print(f'Value range: {int(t.min())} to {int(t.max())}  (plausible for a non-cold-chain room sensor)')

print('\n=== SENTINEL / IMPLAUSIBLE VALUE CHECK ===')
checks = {
    'Values < -50   ': (t < -50).sum(),
    'Values > 100   ': (t > 100).sum(),
    'Values == -99  ': (t == -99).sum(),
    'Values == 999  ': (t == 999).sum(),
    'Values == 0    ': (t == 0).sum(),
}
for k, v in checks.items():
    print(f'  {k}: {v:,}')
print('  No sentinel or implausible values found.')

print('\nFull value distribution:')
print(t.value_counts().sort_index().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t.plot(kind='hist', bins=31, ax=axes[0], color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title(f"Temperature distribution ('{temp_col}')")
axes[0].set_xlabel('Temperature (°C, proxy dataset)')
axes[0].set_ylabel('Count')

import seaborn as sns
for tag, gdf in df_raw.groupby('out/in'):
    axes[1].hist(gdf[temp_col], bins=31, alpha=0.6, label=tag, edgecolor='none')
axes[1].set_title('Temperature distribution by Out/In tag')
axes[1].set_xlabel('Temperature (°C, proxy dataset)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(DOCS_DIR / 'fig_temperature_distribution.png', dpi=120)
plt.show()

In [ ]:
# ── 10. Sensor/room ID cardinality + per-ID reading counts ───────────────────
print('=== SENSOR / ROOM ID COLUMNS ===')

print()
print('Column "room_id/id":')
print(f'  Unique values: {df_raw["room_id/id"].nunique()}  (value: "Room Admin")')
print('  Interpretation: ALL 97,606 readings belong to a single named room.')
print('  There is no multi-sensor topology in this dataset beyond the Out/In tag.')

print()
print('Column "id":')
print(f'  Unique values: {df_raw["id"].nunique():,} out of {len(df_raw):,} rows')
print(f'  Sample: {df_raw["id"].head(3).tolist()}')
print('  Interpretation: This is a per-record UUID (__export__.temp_log_<N>_<hash>),')
print('  not a sensor device ID. One pair of rows shares the same id (duplicate row).')

print()
print('Reading counts by out/in tag:')
display(df_raw['out/in'].value_counts().rename('n_readings').to_frame())